# 04 · Reading a run folder

This notebook reads the records of the run that notebook 01 made: which settings identify the
run (`config.json`), its outcome (`summary.json`), its scores (`scores.jsonl`, `points.csv`).
It recomputes the final held-out NLL and KL by hand, re-scores the final pool from the saved
posterior particles, and compares the run with a second run that differs in one setting only.

- **Needs:** the run folder of [notebook 01](01_engine_basics.ipynb),
  `examples/runs/quickstart_synthcount/`. **Run notebook 01 first**; this notebook stops with a
  message if the folder is missing.
- **Runtime:** about one minute, most of it for the second run in section 7. No API key, no
  cost.
- **Writes:** `examples/runs/quickstart_synthcount_random/` (ignored by git).

The files are described in [`FLOW.md`](../FLOW.md#the-run-folder) and the terms in the
glossary of [notebook 00](00_how_it_works.ipynb).

## 1. Setup

In [1]:
import hashlib
import json
import logging
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import xlogy


def find_repo_root(start):
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "src" / "bayes_model_discovery").is_dir():
            return folder
    raise FileNotFoundError("start Jupyter inside the repository checkout")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
try:
    import bayes_model_discovery  # noqa: F401
except ImportError:  # not installed: use the source tree directly
    sys.path.insert(0, str(REPO / "src"))
logging.getLogger("pytensor.tensor.blas").setLevel(logging.ERROR)
pd.set_option("display.precision", 4)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 60)

RUN = Path("examples/runs/quickstart_synthcount")
if not (RUN / "summary.json").is_file():
    raise FileNotFoundError(
        f"{RUN} does not exist yet. Run notebook 01_engine_basics.ipynb first: "
        "it makes this run folder in a few minutes, with no API key.")
print("run folder:", RUN)

run folder: examples/runs/quickstart_synthcount


## 2. `config.json`: what identifies a run

Read this file first. It says what a run is: the benchmark world (`seed`, `structure`,
`truth_family`, `replicate`), the compute preset (`profile`), the condition code (`cell`), the
system prompt (`prompt_version`, `prompt_sha256`), the engine code (`source_files_sha256`, the
sha256 of every engine file the run loaded) and the package versions. Two runs are comparable
when all of these agree except the one setting under study. Comparing the recorded code
digests with the files of this checkout tells whether the run was made with the code you have.

Some fields hold absolute paths of the machine that made the run (`output`, `python`; in
ChemBench runs also `adapter_source` and others, and `source_path` in `points.csv`). This
notebook prints them as `<path>`; check them before you share a run folder.

In [2]:
config = json.loads((RUN / "config.json").read_text())


def redact(value):
    """Absolute paths of the machine that made the run, replaced before printing."""
    if isinstance(value, str) and (value.startswith("/") or re.match(r"^[A-Za-z]:[\\/]", value)):
        return "<path>"
    return value


for key in ("profile", "seed", "replicate", "structure", "truth_family", "cell", "rounds",
            "design_policy", "proposer", "prompt_version", "prompt_sha256", "model_id_schema",
            "created_utc"):
    print(f"{key:18} {redact(config[key])}")

ENGINE = Path("src/bayes_model_discovery/experimental")
print(f"\nsource_files_sha256: {len(config['source_files_sha256'])} files, three of them:")
for name in ("core_count.py", "count_inference.py", "count_models.py"):
    recorded = config["source_files_sha256"][name]
    current = hashlib.sha256((ENGINE / name).read_bytes()).hexdigest()
    print(f"  {name:19} {recorded}  same as this checkout: {recorded == current}")
print(f"\nversions: {config['versions']}")
print("fields holding an absolute path:", {key: redact(value) for key, value in config.items()
                                         if redact(value) == "<path>"})

profile            dev
seed               11
replicate          0
structure          out
truth_family       beta_binomial
cell               111
rounds             3
design_policy      joint_eig
proposer           mock_fixed_template_responses_NOT_LLM
prompt_version     v2.0
prompt_sha256      f095edf1adb3b6533a9e68e73f98ca1dee3f23cd077e424b092393cb32b93aef
model_id_schema    ast_roles_family_ac_v2
created_utc        2026-10-07T12:53:14.661476+00:00

source_files_sha256: 8 files, three of them:
  core_count.py       eff3d2c5ae49f8fd8f3b360aa8d04004231ad63a694bcc2c8b9b673389dfbd04  same as this checkout: True
  count_inference.py  03d6410b9bb06983b15f8211bb69eb1030b472a3cb25dd95e9e5901ea36323be  same as this checkout: True
  count_models.py     7171ad98ddfeff2b0b2d9cd83b3757cb3d130990fc38cf8734aacf895220a95b  same as this checkout: True

versions: {'numpy': '1.26.4', 'pymc': '5.13.1', 'pytensor': '2.20.0', 'scipy': '1.17.1'}
fields holding an absolute path: {'output': '<path>', 'python'

## 3. `summary.json`: how the run ended

`status` is `completed` only when every round finished; a run that fails writes
`failure.json` instead, and one stopped by a cap or a stop file says why in `stop_reason`.

In [3]:
summary = json.loads((RUN / "summary.json").read_text())
accounting = summary["proposal_accounting"]
print(f"status {summary['status']}; {summary['wall_seconds']:.0f} s wall time; "
      f"{summary['paid_api_calls']} paid API calls ({summary['billing_status']})")
print(f"observations {summary['observations']}, admissions {summary['admissions']}, "
      f"final pool {summary['active_count']}, candidate failures {len(summary['candidate_failures'])}")
print({key: accounting[key] for key in ("proposal_rounds_executed", "valid_candidates", "duplicates",
                                        "prior_probe_rejections", "fit_attempts", "fit_successes")})

status completed; 56 s wall time; 0 paid API calls (no_calls)
observations 7, admissions 3, final pool 7, candidate failures 0
{'proposal_rounds_executed': 3, 'valid_candidates': 3, 'duplicates': 0, 'prior_probe_rejections': 0, 'fit_attempts': 3, 'fit_successes': 3}


## 4. `scores.jsonl`: one line per checkpoint

Each line is one scored **checkpoint**: the round, the scoring phase (`initial`,
`before_proposal`, `after_proposal`), the pool and its evidence weights, and the held-out
scores of the Bayesian model averaging (BMA) predictive (`bma.*`) and of the highest-evidence model alone (`single.*`).
`pd.json_normalize` flattens the nested blocks into columns. Lower NLL and KL are better;
**coverage90** is the fraction of held-out outcomes inside the 90% predictive interval and
**width90** the interval's average width.

In [4]:
rows = [json.loads(line) for line in (RUN / "scores.jsonl").read_text().splitlines() if line.strip()]
print("keys of one line:", ", ".join(rows[0]))
scores = pd.json_normalize(rows)
display(scores[["round", "phase", "observation_count", "active_count", "bma.nll", "bma.kl",
                "bma.coverage90", "bma.width90", "single.nll"]])

keys of one line: active_count, active_ids, bma, model_weights, observation_count, per_model_nll, phase, post_equals_pre, round, scoring_phase, single, single_model_id


,round,phase,observation_count,active_count,bma.nll,bma.kl,bma.coverage90,bma.width90,single.nll
0,0,initial,4,4,5.9577,3.0195,0.4141,12.4375,6.0196
1,1,before_proposal,5,4,4.8434,1.9736,0.4141,12.0000,5.0139
2,1,after_proposal,5,5,3.2588,0.3368,0.9922,34.7500,3.2588
3,2,before_proposal,6,5,3.0844,0.1686,1.0000,32.0312,3.0844
4,2,after_proposal,6,6,3.0844,0.1686,1.0000,32.0312,3.0844
5,3,before_proposal,7,6,3.1170,0.1983,0.9688,32.7500,3.1170
6,3,after_proposal,7,7,3.1170,0.1983,0.9688,32.7500,3.1170


## 5. `points.csv`: the same checkpoints as one table

`points.csv` holds the same checkpoints in a fixed column format that the analysis tools read:
one row per checkpoint, `x` the number of observations so far (the budget) and `y_primary` the
primary score (held-out NLL here). `source_path` points back to the line of `scores.jsonl` the
row came from, and `code_sha` is the sha256 of the runner file. Read it with
`float_precision="round_trip"`: pandas' default float parser can change the last digit, and
then a recomputed score no longer compares equal.

In [5]:
points = pd.read_csv(RUN / "points.csv", float_precision="round_trip")
points["source_path"] = points["source_path"].map(lambda text: "<path>/" + text.rsplit("/", 1)[-1])
print(f"{len(points)} rows; columns: {', '.join(points.columns)}")
display(points[["suite", "task", "method", "seed", "x", "nll", "kl", "source_path"]])
print("code_sha is the sha256 of core_count.py in config.json:",
      bool((points["code_sha"] == config["source_files_sha256"]["core_count.py"]).all()))
print("nll column equals scores.jsonl bma.nll:", points["nll"].tolist() == scores["bma.nll"].tolist())

7 rows; columns: suite, task, method, proposer, seed, x, x_unit, y_primary, y_name, y_label, nll, coverage90, width90, crps, is_0.1, kl, status, label, run_id, source_path, variant, code_sha, brier, nmsle, rmsle


,suite,task,method,seed,x,nll,kl,source_path
0,core,core__structure_out__truth_F2__s11,111,11,4,5.9577,3.0195,"<path>/scores.jsonl#L1(round=0,phase=initial)"
1,core,core__structure_out__truth_F2__s11,111,11,5,4.8434,1.9736,"<path>/scores.jsonl#L2(round=1,phase=before_proposal)"
2,core,core__structure_out__truth_F2__s11,111,11,5,3.2588,0.3368,"<path>/scores.jsonl#L3(round=1,phase=after_proposal)"
3,core,core__structure_out__truth_F2__s11,111,11,6,3.0844,0.1686,"<path>/scores.jsonl#L4(round=2,phase=before_proposal)"
4,core,core__structure_out__truth_F2__s11,111,11,6,3.0844,0.1686,"<path>/scores.jsonl#L5(round=2,phase=after_proposal)"
5,core,core__structure_out__truth_F2__s11,111,11,7,3.1170,0.1983,"<path>/scores.jsonl#L6(round=3,phase=before_proposal)"
6,core,core__structure_out__truth_F2__s11,111,11,7,3.1170,0.1983,"<path>/scores.jsonl#L7(round=3,phase=after_proposal)"


code_sha is the sha256 of core_count.py in config.json: True
nll column equals scores.jsonl bma.nll: True


## 6. Recompute the final score by hand

The last checkpoint is round 3, after the proposal. Two saved arrays are enough to recompute
its scores:

- `predictions/t03_after_proposal.npz` holds `bma_logpmf`, the log predictive probability of
  every count 0 to 50 at each of the 32 held-out inputs;
- `evaluation_scoring_only.npz` holds the held-out inputs `u`, four held-out outcomes `y` per
  input, and `pmf`, the true outcome distribution at each input. This file is used for scoring
  only; nothing in it reaches fitting, design choice or the proposer.

NLL is minus the mean log probability of the held-out outcomes; KL is
Σ_y p_true(y) [log p_true(y) − log p_BMA(y)], averaged over the inputs.

In [6]:
last = rows[-1]
phase = last.get("predictions_from", last["phase"])  # an unchanged pool reuses an earlier array
with np.load(RUN / "predictions" / f"t{last['round']:02d}_{phase}.npz") as saved:
    bma_logpmf = saved["bma_logpmf"]
with np.load(RUN / "evaluation_scoring_only.npz") as heldout:
    u, y, truth = heldout["u"], heldout["y"], heldout["pmf"]
print(f"bma_logpmf {bma_logpmf.shape}, u {u.shape}, y {y.shape}, pmf {truth.shape}")

nll = -float(bma_logpmf[np.arange(len(y))[:, None], y].mean())
kl = float(np.sum(xlogy(truth, truth) - truth * bma_logpmf, axis=-1).mean())
final = points.iloc[-1]
print(f"NLL by hand {nll!r}   points.csv {final['nll']!r}   equal: {nll == final['nll']}")
print(f"KL  by hand {kl!r}   points.csv {final['kl']!r}   equal: {kl == final['kl']}")

bma_logpmf (32, 51), u (32,), y (32, 4), pmf (32, 51)
NLL by hand 3.1170463711140233   points.csv 3.1170463711140233   equal: True
KL  by hand 0.19833785168148382   points.csv 0.19833785168148382   equal: True


**Re-scoring from the posterior.** The saved predictive is itself computed from the saved fits.
Rebuilding the final pool from `fits/t03_*.json` and `.npz` (expression, family, roles, log
evidence, particles) and calling the engine's `count_inference.predict` on the held-out inputs
gives the same predictive distribution, bit for bit, and so the same score. The same route
re-scores a finished run with a different evaluator;
`src/bayes_model_discovery/experimental/frozen_kl_rescore.py` re-scores the last checkpoint of
many finished runs with one fixed evaluator this way.

In [7]:
from bayes_model_discovery.experimental import count_inference
from bayes_model_discovery.experimental.count_models import Hypothesis


def load_fit(folder, t, model_id):
    """Rebuild the engine's Fit object from fits/tNN_<model_id>.json and .npz."""
    record = json.loads((folder / "fits" / f"t{t:02d}_{model_id}.json").read_text())
    hypothesis = Hypothesis(record["expression"], record["family"], tuple(record["parameters"].items()))
    with np.load(folder / "fits" / f"t{t:02d}_{model_id}.npz") as saved:
        particles = {name: saved[name] for name in saved.files}
    return count_inference.Fit(hypothesis, particles, record["log_evidence"], record["per_chain_log_evidence"])


pool = [load_fit(RUN, last["round"], model_id) for model_id in last["active_ids"]]
interface = config["interface"]
prediction = count_inference.predict(pool, u, interface, interface["trials"])
rescored = -float(prediction["bma_logpmf"][np.arange(len(y))[:, None], y].mean())
print(f"pool of {len(pool)} models rebuilt from fits/t{last['round']:02d}_*")
print("same predictive distribution as the saved one:", np.array_equal(prediction["bma_logpmf"], bma_logpmf))
print(f"re-scored NLL {rescored!r}   equal to the recorded one: {rescored == last['bma']['nll']}")

pool of 7 models rebuilt from fits/t03_*
same predictive distribution as the saved one: True
re-scored NLL 3.1170463711140233   equal to the recorded one: True


## 7. Compare two runs

To have a second run to compare with, this cell makes one that differs in one setting: designs
are drawn at random (`--cell 110`) instead of chosen by joint expected information gain (EIG) (`--cell 111`). Same world,
seed, profile and number of rounds. It is skipped when the folder exists.

In [8]:
OTHER = Path("examples/runs/quickstart_synthcount_random")
COMMAND = ["-m", "bayes_model_discovery.experimental.core_count",
           "--profile", "dev", "--proposer", "mock", "--seed", "11", "--cell", "110",
           "--rounds", "3", "--output", str(OTHER)]
print("python " + " ".join(COMMAND) + "\n")
if OTHER.exists():
    print(f"{OTHER} already exists: the run is skipped. Delete the folder to run it again.")
else:
    # Jupyter sets FORCE_COLOR for child processes; the run does not need it (see notebook 01).
    env = {key: value for key, value in os.environ.items() if key not in ("FORCE_COLOR", "CLICOLOR_FORCE")}
    env["PYTHONPATH"] = os.pathsep.join(filter(None, [str(REPO / "src"), env.get("PYTHONPATH")]))
    log_path = OTHER.parent / f"{OTHER.name}.log"
    started = time.perf_counter()
    result = subprocess.run([sys.executable, *COMMAND], capture_output=True, text=True, env=env)
    log_path.write_text(result.stdout + result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"the run failed with exit code {result.returncode}; see {log_path}")
    for line in result.stdout.splitlines():
        if line.startswith("SCORE"):
            print(line)
    print(f"\nfinished in {time.perf_counter() - started:.0f} s; full log: {log_path}")

python -m bayes_model_discovery.experimental.core_count --profile dev --proposer mock --seed 11 --cell 110 --rounds 3 --output examples/runs/quickstart_synthcount_random

SCORE t=0 phase=initial pool=4 NLL=6.132307
SCORE t=1 phase=before_proposal pool=4 NLL=5.424458
SCORE t=1 phase=after_proposal pool=5 NLL=3.260702
SCORE t=2 phase=before_proposal pool=5 NLL=3.308273
SCORE t=2 phase=after_proposal pool=6 NLL=3.308273
SCORE t=3 phase=before_proposal pool=6 NLL=3.132341
SCORE t=3 phase=after_proposal pool=7 NLL=3.132341

finished in 54 s; full log: examples/runs/quickstart_synthcount_random.log


**Do the settings match?** Compare the two `config.json` files key by key, leaving out the
fields that always differ (`created_utc`, `output`, `python`). Only the design rule should
differ.

In [9]:
def load_run(folder):
    config = json.loads((folder / "config.json").read_text())
    scores = pd.json_normalize([json.loads(line) for line in (folder / "scores.jsonl").read_text().splitlines()])
    observations = pd.DataFrame([json.loads(line) for line in (folder / "observations.jsonl").read_text().splitlines()])
    return config, scores, observations


(config_a, scores_a, observations_a), (config_b, scores_b, observations_b) = load_run(RUN), load_run(OTHER)
ALWAYS_DIFFERENT = {"created_utc", "output", "python"}
differ = sorted(key for key in set(config_a) | set(config_b)
                if key not in ALWAYS_DIFFERENT and config_a.get(key) != config_b.get(key))
print(f"{'key':24} {'joint EIG run':24} random-design run")
for key in differ:
    print(f"{key:24} {config_a.get(key)!s:24} {config_b.get(key)}")
shared = ("profile", "seed", "replicate", "structure", "truth_family", "rounds", "prompt_sha256",
          "source_files_sha256", "versions", "evaluation_input_seed", "evaluation_outcome_seed")
print("\nworld, budget, held-out set, prompt, code and versions identical:",
      all(config_a[key] == config_b[key] for key in shared))

key                      joint EIG run            random-design run
cell                     111                      110
design_policy_applies    True                     False
design_policy_target     joint_model_parameter    random

world, budget, held-out set, prompt, code and versions identical: True


**Round 0 already differs.** Both runs fit the same four starting models to the same four
observations, yet their round-0 scores differ (see the table at the end). Each fit's sequential Monte Carlo (SMC) seed
is derived from the world seed, the condition code, the round and the model id
(`seed_for("smc", seed, cell, round, model_id)` in `core_count.py`), so the two runs fit with
different random numbers and get slightly different log evidence. This Monte Carlo noise is part
of every comparison between arms.

In [10]:
round_0 = []
for path in sorted((RUN / "fits").glob("t00_*.json")):
    a, b = json.loads(path.read_text()), json.loads((OTHER / "fits" / path.name).read_text())
    round_0.append({"expression": a["expression"], "log evidence (111)": a["log_evidence"],
                    "log evidence (110)": b["log_evidence"], "SMC seed (111)": a["diagnostics"]["seed"],
                    "SMC seed (110)": b["diagnostics"]["seed"]})
display(pd.DataFrame(round_0))

,expression,log evidence (111),log evidence (110),SMC seed (111),SMC seed (110)
0,b0+b1*z+b2*square(z),-17.3678,-16.6701,1077236846,3307592045
1,b0+b1*log(1+u),-26.2996,-26.4565,1861484502,6693474
2,b0+b1*z,-24.6528,-24.7877,3489588349,1012302596
3,b0,-33.9357,-33.8571,2708962628,3796696198


**Designs and outcomes.** Both runs start from the same four initial-design (D0) observations; then each
chooses its own designs. The outcome of an observation is drawn with a seed that depends on the
world, the design and the repetition, not on the condition code, so two runs that observe the
same design get the same outcome there. That makes runs of different arms easier to compare.

In [11]:
later = pd.DataFrame({
    "joint EIG: u": observations_a.set_index("round")["u"],
    "joint EIG: y": observations_a.set_index("round")["y"],
    "random: u": observations_b.set_index("round")["u"],
    "random: y": observations_b.set_index("round")["y"],
}).loc[1:]
display(later)
pairs = observations_a.merge(observations_b, on=["design_id", "repetition"], suffixes=("_a", "_b"))
print(f"observations at the same design and repetition in both runs: {len(pairs)}; "
      f"same outcome every time: {bool((pairs['y_a'] == pairs['y_b']).all())}")

,joint EIG: u,joint EIG: y,random: u,random: y
round,,,,
1,1.0,47,0.2000,0
2,0.0,0,0.4667,1
3,1.0,27,1.0000,47


observations at the same design and repetition in both runs: 5; same outcome every time: True


**Scores side by side**, for each round and scoring phase:

In [12]:
side_by_side = pd.concat({
    "joint EIG (111)": scores_a.set_index(["round", "phase"])[["bma.nll", "bma.kl"]],
    "random (110)": scores_b.set_index(["round", "phase"])[["bma.nll", "bma.kl"]],
}, axis=1)
display(side_by_side)

joint EIG (111)         random (110)        
                              bma.nll  bma.kl      bma.nll  bma.kl
round phase                                                       
0     initial                  5.9577  3.0195       6.1323  3.2399
1     before_proposal          4.8434  1.9736       5.4245  2.5760
      after_proposal           3.2588  0.3368       3.2607  0.3238
2     before_proposal          3.0844  0.1686       3.3083  0.3782
      after_proposal           3.0844  0.1686       3.3083  0.3782
3     before_proposal          3.1170  0.1983       3.1323  0.2250
      after_proposal           3.1170  0.1983       3.1323  0.2250

One seed, three rounds and scripted proposals cannot say which design rule is better; this
only shows how to line two runs up. The paper compares the five design criteria (four on
ChemBench) over three seeds and 30 rounds (16 on ForceBench, 5 on NeuronBench); see
[`FLOW.md`](../FLOW.md#the-protocol-of-the-paper-runs).